AnnData is a python data structure for annotated data. It is an object including fields of the raw data, metadata per each observation, and additional information. It is stored in h5ad files which can be read with scanpy.

This is a mutable structure and changes to slices reference the original structure. Use copy() method to create independent structures.

Notable fields in the context of scRNA-seq data: 
- X: contains the raw data with shape of n_obs (the cells) and n_vars (the genes) stored in a compressed sparse row matrix
    - Size of X is 18400 (46 non-targeting guide RNAs * 400 cells), 18533 (number of genes in gene_names.csv)
    - The compressed sparse row matrix representation allows for memory efficiency as most scRNA-data is zero values
- Shape: returns the shape (num rows, num columns) of X as tuple
- Obs: contains metadata for each row (cell) specifically the cell's identifier, target_gene of the guide RNA, and the non-targeting control guide rna identifier. 
- Var: contains metadata for each column (gene). The only value here is the gene name.
- Obsm: contains multi-dimensional observations. Nothing here for these datasets.
- Varm: contains the annotations on multi-dimensional observations. Nothing here for these datasets. 
- Obsp: contains pairwise observation matrices. Nothing here for these datasets
- Varp: contains pairwise variable matrices. Nothing here for these datasets.
- Layers: contains alternative representations of X. Nothing here for these datasets.
- Uns: unstructure annotations.

CSR matrices only include non-zero values and are objects consisting of three arrays: data, indices, indptr (indices pointer). 

All the data across all rows and columns is stored linearly in data. The length of data is equivalent to the lenght of indices, as indices specifices which column the value in data pertains to. The boundaries of rows is indicated by indptr where data[indptr[i] to indptr[i+1]] exclusive is row i's data. 

In [2]:
import scanpy as sc
import pandas as pd
import anndata as ad
import numpy as np
from scipy.sparse import csr_matrix

In [3]:
cell_contexts = ["A", "B", "C"]
context_adata = dict.fromkeys(cell_contexts)

for key in cell_contexts:
    context_adata[key] = sc.read_h5ad(f"context_{key}.h5ad")

print(context_adata)

{'A': AnnData object with n_obs × n_vars = 18400 × 18533
    obs: 'target_gene', 'context', 'ntc_id', 'B': AnnData object with n_obs × n_vars = 18400 × 18533
    obs: 'target_gene', 'context', 'ntc_id', 'C': AnnData object with n_obs × n_vars = 18400 × 18533
    obs: 'target_gene', 'context', 'ntc_id'}


In [4]:
for context, adata in context_adata.items():
    print("_______________________")
    print(adata.obs) 
    print("_______________________")
    print(adata.var)
    print("_______________________")
    print(adata.obsm)
    print("_______________________")
    print(adata.varm)
    print("_______________________")
    print(adata.obsp)
    print("_______________________")
    print(adata.varp)
    print("_______________________")
    print(adata.layers)
    print("_______________________")

_______________________
                 target_gene context            ntc_id
A_ctrl_000000  non-targeting       A  non-targeting-11
A_ctrl_000001  non-targeting       A  non-targeting-11
A_ctrl_000002  non-targeting       A  non-targeting-11
A_ctrl_000003  non-targeting       A  non-targeting-11
A_ctrl_000004  non-targeting       A  non-targeting-11
...                      ...     ...               ...
A_ctrl_018395  non-targeting       A  non-targeting-99
A_ctrl_018396  non-targeting       A  non-targeting-99
A_ctrl_018397  non-targeting       A  non-targeting-99
A_ctrl_018398  non-targeting       A  non-targeting-99
A_ctrl_018399  non-targeting       A  non-targeting-99

[18400 rows x 3 columns]
_______________________
Empty DataFrame
Columns: []
Index: [TSPAN6, TNMD, DPM1, SCYL3, C1orf112, FGR, CFH, FUCA2, GCLC, NFYA, STPG1, NIPAL3, LAS1L, ENPP4, SEMA3F, CFTR, ANKIB1, CYP51A1, KRIT1, RAD52, BAD, LAP3, CD99, HS3ST1, AOC1, WNT16, HECW1, MAD1L1, LASP1, SNX11, TMEM176A, M6PR, KLHL13,

In [5]:
for context, adata in context_adata.items():
    # fields of the adata.X object which is a CSR matrix
    print(f'Non-zero entries: {adata.X.nnz} from total entries {adata.shape[0]*adata.shape[1]}')
    # print(adata.X.data)
    # print(adata.X.indices)
    # print(adata.X.indptr)

Non-zero entries: 109910478 from total entries 341007200
Non-zero entries: 101546777 from total entries 341007200
Non-zero entries: 108213983 from total entries 341007200


In [6]:
for context, adata in context_adata.items():
    n_cells_data = adata.X[:10].toarray() # don't use this toarray method for large amounts of data
    print(pd.DataFrame(n_cells_data, index=adata.obs_names[:10], columns=adata.var_names))

               TSPAN6  TNMD  DPM1  SCYL3  C1orf112  FGR  CFH  FUCA2  GCLC  \
A_ctrl_000000     0.0   0.0   5.0    0.0       3.0  0.0  0.0    0.0   1.0   
A_ctrl_000001     0.0   0.0   3.0    1.0       2.0  0.0  0.0    0.0   3.0   
A_ctrl_000002     0.0   0.0   7.0    0.0       7.0  0.0  0.0    0.0   0.0   
A_ctrl_000003     0.0   0.0   3.0    1.0       0.0  0.0  0.0    0.0   5.0   
A_ctrl_000004     0.0   0.0   4.0    0.0       3.0  0.0  0.0    0.0   1.0   
A_ctrl_000005     0.0   0.0   4.0    1.0       3.0  0.0  0.0    0.0   8.0   
A_ctrl_000006     0.0   0.0   4.0    3.0       1.0  1.0  0.0    0.0   2.0   
A_ctrl_000007     0.0   0.0   3.0    0.0       2.0  0.0  0.0    0.0   4.0   
A_ctrl_000008     0.0   0.0   3.0    0.0       5.0  0.0  0.0    0.0   2.0   
A_ctrl_000009     0.0   0.0   1.0    0.0       1.0  0.0  0.0    0.0   0.0   

               NFYA  ...  AC253572.1  AL603764.2  DERPC  AC023490.5  \
A_ctrl_000000   1.0  ...         0.0         0.0    0.0         0.0   
A_ctrl_000

In [10]:
# Dummy prediction h5ad with correct structure but nearly all zeros
from scipy.sparse import lil_matrix

gene_names_list = pd.read_csv("gene_names.csv")["gene_name"].tolist()
pert_genes_list = pd.read_csv("pert_counts.csv")["target_gene"].tolist()
contexts = ["A", "B", "C"]
cells_per_pert = 400
n_genes = len(gene_names_list)
n_total = len(pert_genes_list) * cells_per_pert * len(contexts)  # 360,000

# Build obs
obs_rows = []
for ctx in contexts:
    for gene in pert_genes_list:
        obs_rows.extend([{"target_gene": gene, "context": ctx}] * cells_per_pert)
obs = pd.DataFrame(obs_rows)
obs.index = [str(i) for i in range(n_total)]

# Sparse X: all zeros except a 1 in the first gene of every 100th cell
X = lil_matrix((n_total, n_genes), dtype=np.float32)
for i in range(0, n_total, 100):
    X[i, 0] = 1.0
X = X.tocsr()

dummy = ad.AnnData(X=X, obs=obs, var=pd.DataFrame(index=gene_names_list))
dummy.write_h5ad("dummy_prediction.h5ad")

print(f"Shape: {dummy.shape}")
print(f"nnz: {dummy.X.nnz}")
print(f"obs columns: {list(dummy.obs.columns)}")
print(dummy.obs.head())
print("\nSaved to dummy_prediction.h5ad")

Shape: (360000, 18533)
nnz: 3600
obs columns: ['target_gene', 'context']
  target_gene context
0       ABCD1       A
1       ABCD1       A
2       ABCD1       A
3       ABCD1       A
4       ABCD1       A

Saved to dummy_prediction.h5ad


In [ ]:
from scipy.sparse import vstack
import gc

gene_names = pd.read_csv("gene_names.csv")["gene_name"].tolist()
pert_genes = pd.read_csv("pert_counts.csv")["target_gene"].tolist()

contexts = ["A", "B", "C"]
cells_per_pert = 400
n_perts = len(pert_genes)
rng = np.random.default_rng(42)

# Baseline: for each perturbation, sample 400 random control cells from that context.
# This predicts "the perturbation has no effect" while preserving realistic sparsity.
X_blocks = []
obs_rows = []

for ctx in contexts:
    ctrl = sc.read_h5ad(f"context_{ctx}.h5ad")
    n_ctrl = ctrl.X.shape[0]
    print(f"Context {ctx}: sampling {cells_per_pert} control cells for each of {n_perts} perturbations...")

    flat_idx = rng.integers(0, n_ctrl, size=n_perts * cells_per_pert)
    X_blocks.append(ctrl.X[flat_idx].copy())

    for gene in pert_genes:
        obs_rows.extend([{"target_gene": gene, "context": ctx}] * cells_per_pert)

    del ctrl, flat_idx
    gc.collect()
    print(f"  Block shape: {X_blocks[-1].shape}, nnz: {X_blocks[-1].nnz:,}")

print("Stacking context blocks...")
X = vstack(X_blocks, format='csr')
del X_blocks
gc.collect()

obs = pd.DataFrame(obs_rows)
obs.index = [str(i) for i in range(len(obs))]

adata = ad.AnnData(X=X, obs=obs, var=pd.DataFrame(index=gene_names))

print(f"\nShape: {adata.shape}")
print(f"obs columns: {list(adata.obs.columns)}")
print(adata.obs.head())
print(f"\nNon-zero entries: {adata.X.nnz:,} / {adata.shape[0] * adata.shape[1]:,}")
print(f"Sample values (first cell): {adata.X[0, :10].toarray().flatten()}")
print(f"\n... {len(adata.obs)} total cells")

adata.write_h5ad("sampled_baseline_prediction.h5ad")
print("\nSaved to sampled_baseline_prediction.h5ad")